# Глава 1. Введение в RAG-системы

## 1.1. Что такое RAG и зачем он нужен

Проблема LLM - её знания ограниченны данными, на которых ее обучили.

**RAG (Retrieval-Augmented Generation)** - дополненная поиском генерация - архитектурный подход, который соединяет LLM с внешней, постоянно обовляемой базой знаний.

LLM страдают от нескольких критических ограничений:
- **Галлюцинации** - когда модель сообшает ложную информацию, пытается дать ответ на любой вопрос, даже если у неё нет достоверных данных по теме;
- Устаревание данных - модель обучена на данных до 2023 года, она не знает о событиях 2024 года;
- Отсутствие спициализированных знаний. Общие модели хорошо справляются с популярными темами, но могут давать поверхностные или неточные ответы по узкоспециализированным вопросам: внутренним процессам компании, отраслевым регламентам, техническим спецификациям оборудования.

RAG дает возможность динамически обращаться к внешним источникам в момент формирования ответа:
- Когда поступает запрос пользователя, RAG‑система сначала преобразует его в векторное представление и ищет наиболее релевантные фрагменты в предварительно проиндексированной базе знаний;
- Найденная информация передаётся языковой модели как дополнительный контекст, на основе которого формируется итоговый ответ.

 ---

**RAG дополняет промпт** (или, точнее, контекст, в котором работает модель).

Вот как это работает на практике:

1.  **Без RAG:** Модель отвечает только на основе того, чему она научилась во время обучения. Промпт состоит только из инструкции и вопроса пользователя.
    *   *Запрос:* «Какие планы у компании X на следующий квартал?»
    *   *Ответ модели:* «Простите, у меня нет актуальных данных (знания заканчиваются на дату...).»

2.  **С RAG:** В исходный промпт вклинивается дополнительный блок — добытый релевантный контекст.
    *   *Сценарий:* Система нашла в базе внутренний документ о планах.
    *   *Фактический промпт, который получает модель:*
        > «Ответь на вопрос пользователя, используя **ТОЛЬКО** приведенный ниже контекст. Если в контексте нет ответа, скажи, что не знаешь.
        >
        > **Контекст:** "Согласно внутренней дорожной карте, компания X планирует запустить новую линейку продуктов в 3-м квартале."
        >
        > **Вопрос пользователя:** Какие планы у компании X на следующий квартал?»

**Главное отличие от обычного дополнения промпта:**
Обычный промпт — это статическая инструкция от разработчика. RAG-контекст — это **динамические данные**, которые извлекаются из базы знаний в реальном времени специально под конкретный запрос.

Таким образом, LLM получает не просто «дополненный промпт», а **собранный на лету контекстный пакет**, что и позволяет ей давать актуальные, фактологически точные ответы без переобучения модели.

---

Ключевое преимущество — разделение хранения знаний и их обработки. Мы можем обновлять базу знаний в режиме реального времени, добавляя новые документы, корректируя устаревшие данные, интегрируя отраслевую специфику — и всё это без необходимости переобучения LLM.

RAG — не панацея от всех проблем языковых моделей. Качество ответов напрямую зависит от качества данных в базе знаний. Если исходные документы содержат противоречивую или неточную информацию, система может усиливать эти проблемы.

RAG также не полностью устраняет галлюцинации — модель всё ещё может творчески интерпретировать найденную информацию или заполнять пробелы собственными «додумываниями». Именно поэтому современные RAG‑системы дополняются механизмами контроля качества и проверки фактов.

## 1.2. Эволюция от поисковых систем

Первое поколение поисковых технологий базировалось на принципе прямого соответствия слов. Подход TF‑IDF (Term Frequency‑Inverse Document Frequency), появившийся в 1970‑х годах, стал фундаментом информационного поиска на десятилетия вперёд. Его гениальность заключалась в простоте: часто встречающиеся в документе слова получают высокий вес, а редкие в коллекции термины считаются более значимыми.

TF‑IDF имел критический недостаток — проблему масштабирования по длине документов. Два документа разной длины с одинаковым содержанием получали различные оценки релевантности, что искажало результаты поиска. 

Алгоритм BM25 (Best Matching 25) -  ввёл концепцию насыщения частоты терминов — идею о том, что увеличение количества вхождений слова в документ даёт убывающую отдачу. Если в статье о Ломоносове его имя упоминается 12 раз вместо шести, это не делает статью в два раза более релевантной — закон убывающей полезности действует и в информационном поиске.

Семантическая революция началась с появлением искусственных нейронных сетей, способных преобразовывать слова в числовые векторы — эмбеддинги.

Технологии Word2Vec, а позднее BERT и их наследники позволили кодировать смысловое содержание текста в многомерном пространстве, в котором семантически близкие понятия располагаются рядом.

Векторный поиск кардинально изменил правила игры. Вместо поиска точных совпадений система начала искать семантическое сходство между запросом и документами. Запрос «найти информацию о покупке автомобиля» мог найти документы со словами «приобретение», «машина», «транспорт» — поисковая система научилась понимать смысл, а не только буквы.

Ключевым преимуществом стала способность поисковых систем работать с запросами на разговорном языке. Пользователь мог задать вопрос «Как лучше выбрать подержанную машину?» и получить релевантные результаты, даже если в найденных документах не было точного совпадения фразы.

Векторный поиск превосходно работает с концептуальными и абстрактными запросами, но может «терять» точные совпадения имён, артикулов, специальных терминов.

Современные системы используют гибридный подход, комбинируя алгоритм BM25 для точных лексических совпадений с векторным поиском для семантического понимания. Такая архитектура обеспечивает как точность в поиске конкретной информации, так и гибкость в понимании пользовательских интенций.

Следующий эволюционный скачок произошёл с появлением больших языковых моделей и концепции RAG. Если традиционные поисковые системы могли только найти и ранжировать документы, то RAG‑системы способны понять найденную информацию и синтезировать осмысленный ответ.

RAG объединил три ключевых компонента:
- продвинутый поисковый движок (часто гибридный);
- базу знаний с возможностью динамического обновления;
- генеративную языковую модель, способную создавать связные ответы на основе найденной информации.

Это позволило перейти от парадигмы «найти документы» к парадигме «получить ответ».

Современные RAG‑системы представляют собой полноценные интеллектуальные платформы, способные не только искать информацию, но и рассуждать, анализировать противоречия, учитывать контекст предыдущих взаимодействий. Они могут работать с мультимодальным контентом — текстами, изображениями, структурированными данными, предоставляя пользователю единый интерфейс для работы с разнородной информацией.

Ключевое отличие современных RAG‑систем от классических поисковых машин заключается в способности к диалоговому взаимодействию. Система не только находит ответ на прямой вопрос, но может уточнить неясные моменты, предложить альтернативные интерпретации запроса, объяснить свою логику.